# Pure-State Markov Snapshot Runs

Runs four canonical GPU Markov-circuit configurations and saves sparse covariance snapshots at cycles 5, 10, 20, and 50.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_small_system_testing'

def is_bundle_root(path: Path) -> bool:
    return (path / 'src' / 'classA_U1FGTN_gpu.py').exists()

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])
if IN_COLAB:
    candidates.extend([
        Path('/content/drive/MyDrive') / BUNDLE_NAME,
        Path('/content/drive/MyDrive/class_A_fermionic_adaptive_circuit') / BUNDLE_NAME,
    ])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if is_bundle_root(candidate):
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME} with src/classA_U1FGTN_gpu.py')

SRC_DIR = BUNDLE_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'[setup] bundle root: {BUNDLE_ROOT}')

In [ ]:
import numpy as np
from tqdm.auto import tqdm
import torch

from classA_U1FGTN_gpu import classA_U1FGTN_gpu

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this Colab GPU campaign.')

print(f'[setup] torch={torch.__version__}')
print(f'[setup] cuda device: {torch.cuda.get_device_name(0)}')

In [ ]:
NX = 16
NY_VALUES = [30, 40]
NSHELL_VALUES = [1, 2]
SAMPLES = 10
CYCLES = 50
SNAPSHOT_CYCLES = [5, 10, 20, 50]
ALPHA_1 = 1.0
ALPHA_2 = 30.0
DW_TRUNCATION = True
PROTOCOL = 'perfect_correction'
INIT_MODE = 'default'
DTYPE = 'complex128'
DEVICE = 'cuda:0'
BACKEND = 'local'
SEQUENCE = 'raster_y'
BATCH_SIZE = None

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
CAMPAIGN_NAME = 'pure_state_covariance_snapshots'
OUTPUT_ROOT = GPU_DATA_ROOT / CAMPAIGN_NAME
RUNS_ROOT = OUTPUT_ROOT / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

CONFIGS = [
    {'Nx': NX, 'Ny': int(ny), 'nshell': int(nshell)}
    for ny in NY_VALUES
    for nshell in NSHELL_VALUES
]

CANONICAL_ENTRY_POINT = 'classA_U1FGTN_gpu.run_markov_circuit'

print(json.dumps({
    'configs': CONFIGS,
    'samples': SAMPLES,
    'cycles': CYCLES,
    'snapshot_cycles': SNAPSHOT_CYCLES,
    'dtype': DTYPE,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'output_root': str(OUTPUT_ROOT),
    'runs_root': str(RUNS_ROOT),
}, indent=2))


In [ ]:
def write_json_atomic(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + '.tmp')
    with tmp_path.open('w', encoding='utf-8') as fh:
        json.dump(payload, fh, indent=2, sort_keys=True)
    tmp_path.replace(path)


def rel_to_gpu_data(path) -> str | None:
    if path is None:
        return None
    try:
        return str(Path(path).resolve().relative_to(GPU_DATA_ROOT.resolve()))
    except Exception:
        return None


def update_gpu_data_index() -> None:
    index_path = GPU_DATA_ROOT / 'index.json'
    payload = {
        'bundle_root': str(BUNDLE_ROOT),
        'gpu_data_root': str(GPU_DATA_ROOT),
        'campaigns': [
            {
                'name': 'pure_state_covariance_snapshots',
                'kind': 'raw_covariance_snapshots',
                'manifest': 'pure_state_covariance_snapshots/campaign_manifest.json',
            },
            {
                'name': 'purification_entropy_contours_maxmix',
                'kind': 'derived_entropy_observable',
                'manifest': 'purification_entropy_contours_maxmix/latest_campaign.json',
            },
        ],
    }
    write_json_atomic(index_path, payload)


def case_output_dir(cfg: dict) -> Path:
    return RUNS_ROOT / f"N{cfg['Nx']}x{cfg['Ny']}_nsh{cfg['nshell']}_{PROTOCOL}"


def storage_gib_for_config(cfg: dict) -> float:
    nlayer = 2 * int(cfg['Nx']) * int(cfg['Ny'])
    byte_count = SAMPLES * len(SNAPSHOT_CYCLES) * nlayer * nlayer * np.dtype(np.complex128).itemsize
    return byte_count / 1024**3


def run_one_config(cfg: dict) -> dict:
    out_dir = case_output_dir(cfg)
    out_dir.mkdir(parents=True, exist_ok=True)
    print(
        f"[config] Nx={cfg['Nx']} Ny={cfg['Ny']} nshell={cfg['nshell']} "
        f"snapshots={SNAPSHOT_CYCLES} dtype={DTYPE} output={out_dir}"
    )

    model = classA_U1FGTN_gpu(
        Nx=int(cfg['Nx']),
        Ny=int(cfg['Ny']),
        DW=True,
        nshell=int(cfg['nshell']),
        filling_frac=0.5,
        alpha_1=ALPHA_1,
        alpha_2=ALPHA_2,
        trial_orbitals='X',
        dw_truncation=DW_TRUNCATION,
        triv_region_local_mode=False,
        device=DEVICE,
        dtype=DTYPE,
        backend=BACKEND,
    )

    result = model.run_markov_circuit(
        G_history=False,
        progress=False,
        cycles=CYCLES,
        postselect=False,
        perfect_correction=True,
        samples=SAMPLES,
        init_mode=INIT_MODE,
        save=True,
        save_init=False,
        save_history_stride=None,
        snapshot_cycles=SNAPSHOT_CYCLES,
        save_suffix='snapshots',
        n_a=0.5,
        sequence=SEQUENCE,
        batch_size=BATCH_SIZE,
        save_dir=str(out_dir),
        resume=True,
        merge_on_finish=False,
        return_data=False,
        run_dir_style='compact',
    )

    summary = {
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'nshell': int(cfg['nshell']),
        'samples': SAMPLES,
        'cycles': CYCLES,
        'snapshot_cycles': SNAPSHOT_CYCLES,
        'protocol': PROTOCOL,
        'perfect_correction': True,
        'init_mode': INIT_MODE,
        'alpha_1': ALPHA_1,
        'alpha_2': ALPHA_2,
        'dw_truncation': DW_TRUNCATION,
        'dtype_requested': DTYPE,
        'dtype_resolved': 'complex128',
        'device': DEVICE,
        'backend_requested': BACKEND,
        'backend_resolved': str(model.backend),
        'sequence': SEQUENCE,
        'batch_size_requested': BATCH_SIZE,
        'batch_size': int(result['batch_size']),
        'batch_size_mode': result['batch_size_mode'],
        'batch_size_auto_info': result.get('batch_size_auto_info'),
        'estimated_snapshot_storage_gib': round(storage_gib_for_config(cfg), 4),
        'run_dir': str(result['run_dir']),
        'manifest_path': str(result['manifest_path']),
        'save_path': str(result['save_path']),
        'run_dir_relative': rel_to_gpu_data(result['run_dir']),
        'manifest_path_relative': rel_to_gpu_data(result['manifest_path']),
        'save_path_relative': rel_to_gpu_data(result['save_path']),
    }
    summary_path = Path(result['run_dir']) / 'run_summary.json'
    summary['summary_path'] = str(summary_path)
    summary['summary_path_relative'] = rel_to_gpu_data(summary_path)
    write_json_atomic(summary_path, summary)
    latest_path = out_dir / 'latest_run.json'
    write_json_atomic(latest_path, {'run_summary_path': str(summary_path), 'run_summary_path_relative': rel_to_gpu_data(summary_path), **summary})
    return summary


In [ ]:
all_results = []
for cfg in tqdm(CONFIGS, desc='Configurations', unit='config'):
    all_results.append(run_one_config(cfg))

campaign_manifest = {
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'bundle_root': str(BUNDLE_ROOT),
    'gpu_data_root': str(GPU_DATA_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'runs_root': str(RUNS_ROOT),
    'output_root_relative': rel_to_gpu_data(OUTPUT_ROOT),
    'runs_root_relative': rel_to_gpu_data(RUNS_ROOT),
    'configs': CONFIGS,
    'results': all_results,
    'samples': SAMPLES,
    'cycles': CYCLES,
    'snapshot_cycles': SNAPSHOT_CYCLES,
    'protocol': PROTOCOL,
    'dtype': DTYPE,
}
campaign_manifest_path = OUTPUT_ROOT / 'campaign_manifest.json'
write_json_atomic(campaign_manifest_path, campaign_manifest)
update_gpu_data_index()
print(f'[done] campaign manifest: {campaign_manifest_path}')


In [ ]:
if IN_COLAB:
    from google.colab import runtime  # type: ignore
    runtime.unassign()
else:
    print('[done] not running in Colab; runtime disconnect skipped')